In [5]:
import torch

# ==========================================
# 1. 定义 MDP 核心要素
# ==========================================

# 【状态转移矩阵 T】
# 严格遵循原文的 T(s, a, s')，维度为 [3个状态, 2个动作, 3个下一状态]
# 状态：0=study, 1=sleep, 2=play games
# 动作：0=work, 1=slack
T = torch.tensor([
    # 从 s0 (study) 出发
    [[0.8, 0.1, 0.1],  # 执行 a0 (work)：80%留在study，10%去sleep，10%去play
     [0.1, 0.6, 0.3]], # 执行 a1 (slack)：10%去study，60%留在sleep，30%去play
    
    # 从 s1 (sleep) 出发
    [[0.7, 0.2, 0.1],  # 执行 a0 (work)
     [0.1, 0.8, 0.1]], # 执行 a1 (slack)
    
    # 从 s2 (play games) 出发
    [[0.6, 0.2, 0.2],  # 执行 a0 (work)
     [0.1, 0.4, 0.5]]  # 执行 a1 (slack)
])

# 【奖励函数 R】
# 长度为 3，对应 3 个状态的即时奖励：study(+1), sleep(0), play(-1)
R = torch.tensor([1., 0, -1.])

# 【折扣因子 gamma】
# 控制对未来奖励的重视程度
gamma = 0.5

# 【最优策略】
# 原文设定：在所有状态下都选择 a0 (work)
action = 0


# ==========================================
# 2. 核心算法：矩阵求逆法求解价值函数
# ==========================================

def cal_value_matrix_inversion(gamma, trans_matrix, rewards):
    """
    利用贝尔曼期望方程的解析解计算状态价值函数 V
    数学公式: V = (I - gamma * T)^(-1) * R
    """
    n_states = rewards.shape[0]  # 状态数量 (3)
    
    # 构建单位矩阵 I，并计算 (I - gamma * T)
    I = torch.eye(n_states)
    matrix_to_invert = I - gamma * trans_matrix
    
    # 求逆矩阵
    inv = torch.inverse(matrix_to_invert)
    
    # 将奖励向量 R 转为列向量 (3x1)，进行矩阵乘法
    V = torch.mm(inv, rewards.reshape(-1, 1))
    
    return V


# ==========================================
# 3. 提取当前策略下的转移矩阵并计算
# ==========================================

# 因为策略是固定选择 action=0 (work)，
# 所以提取 T 中所有状态在动作 0 下的转移概率，得到一个 3x3 的方阵
# trans_matrix[i][j] 表示：在状态 i 执行 work，转移到状态 j 的概率
trans_matrix = T[:, action, :]

print("提取的 3x3 转移矩阵 (所有状态执行 work 动作):\n", trans_matrix)
print("-" * 50)

# 测试不同的折扣因子 gamma
gamma = 0.5
V = cal_value_matrix_inversion(gamma, trans_matrix, R)
print(f"gamma={gamma} 时的状态价值函数 V:\n{V}\n")

gamma = 0
V = cal_value_matrix_inversion(gamma, trans_matrix, R)
print(f"gamma={gamma} 时的状态价值函数 V (只看即时奖励):\n{V}\n")

gamma = 0.99
V = cal_value_matrix_inversion(gamma, trans_matrix, R)
print(f"gamma={gamma} 时的状态价值函数 V (极度重视未来):\n{V}\n")

提取的 3x3 转移矩阵 (所有状态执行 work 动作):
 tensor([[0.8000, 0.1000, 0.1000],
        [0.7000, 0.2000, 0.1000],
        [0.6000, 0.2000, 0.2000]])
--------------------------------------------------
gamma=0.5 时的状态价值函数 V:
tensor([[ 1.6787],
        [ 0.6260],
        [-0.4820]])

gamma=0 时的状态价值函数 V (只看即时奖励):
tensor([[ 1.],
        [ 0.],
        [-1.]])

gamma=0.99 时的状态价值函数 V (极度重视未来):
tensor([[65.8294],
        [64.7195],
        [63.4877]])



### 关键修正说明：
1. **维度对齐**：将 `T` 的定义从 `[3, 2, 3]` 修正为 `[状态数, 动作数, 下一状态数]`，这样 `T[:, action, :]` 提取出来的就是完美的 `3x3` 方阵，与 `R` 的长度 3 完全匹配。
2. **数学严谨性**：矩阵求逆公式 $V = (I - \gamma P)^{-1} R$ 要求 $P$ 必须是方阵（状态数 = 下一状态数），修正后的代码在数学上完全成立。
3. **物理意义清晰**：现在你可以明确看到，`trans_matrix` 的每一行代表在 `study`、`sleep`、`play` 三个状态下，执行 `work` 动作后转移到其他三个状态的概率。